# Qwen3-VL ハンズオン: 画像 + テキスト入力で KV cache を使う

このNotebookでは、**同じ画像を何度も見せずに、共有のマルチモーダルprefixを一度だけ前計算し、そのKV cacheを再利用する**流れを確認します。

典型ユースケース:
- 同じ画像に対して複数の質問をしたい
- 最初の重いprefillを再利用して後続質問を軽くしたい
- 画像+テキストの共通コンテキストを何度も使いたい

## 0. KV cache の考え方

自己回帰生成では、過去トークンの attention 用 Key/Value を毎回計算すると無駄が出ます。
そこで一度計算したものを `past_key_values` として保持し、次のデコードで再利用します。

Qwen3-VLでも基本は同じです。違うのは、**共有prefixの中に画像トークンが含まれる**ことです。
今回は次の手順で進めます:

1. 共有prefix = `system + image + 説明文` を作る
2. そのprefixを `use_cache=True` で一度forwardする
3. 得られた `past_key_values` を保存する
4. 各質問ごとに、**prefix以降の差分トークンだけ**を流して generate する

In [ ]:
from copy import deepcopy
from pathlib import Path
from time import perf_counter

import requests
from PIL import Image
import matplotlib.pyplot as plt
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
IMAGE_URL = "https://qianwen-res.oss-cn-beijing.aliyuncs.com/Qwen-VL/assets/demo.jpeg"
IMAGE_PATH = Path("../data/demo.jpeg")
IMAGE_PATH.parent.mkdir(parents=True, exist_ok=True)

if not IMAGE_PATH.exists():
    response = requests.get(IMAGE_URL, timeout=60)
    response.raise_for_status()
    IMAGE_PATH.write_bytes(response.content)

image = Image.open(IMAGE_PATH).convert("RGB")
plt.figure(figsize=(8, 5))
plt.imshow(image)
plt.axis("off")
plt.show()

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if device == "cuda" else torch.float32

model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=dtype, device_map="auto")
processor = AutoProcessor.from_pretrained(MODEL_ID)
print({"device": device, "dtype": str(dtype)})

## 1. 共有prefixを作る

ここで画像を含む共通コンテキストを作ります。
後続の質問はこのprefixを共有します。

In [ ]:
prefix_messages = [
    {
        "role": "system",
        "content": [{"type": "text", "text": "You are a precise visual assistant."}],
    },
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "Understand the image carefully. I will ask multiple follow-up questions about the same image."},
        ],
    },
]

prefix_inputs = processor.apply_chat_template(
    prefix_messages,
    tokenize=True,
    add_generation_prompt=False,
    return_dict=True,
    return_tensors="pt",
)
prefix_inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in prefix_inputs.items()}

prefix_len = prefix_inputs["input_ids"].shape[1]
prefix_len

## 2. 共有prefixを一度だけprefillして KV cache を得る

In [ ]:
t0 = perf_counter()
with torch.inference_mode():
    prefix_outputs = model(**prefix_inputs, use_cache=True, return_dict=True)
prefill_seconds = perf_counter() - t0

base_cache = prefix_outputs.past_key_values
print(f"prefix tokens: {prefix_len}")
print(f"prefill seconds: {prefill_seconds:.2f}")
print(type(base_cache))

## 3. 各質問で差分だけ流す

ポイントは、各質問の**完全入力**を一度作ってから、
`prefix_len` 以降だけを `delta_ids` として切り出すことです。

これにより、共有画像prefixの再計算を避けられます。

In [ ]:
questions = [
    "What is the dog doing?",
    "What clues suggest the setting is calm and pleasant?",
    "List 3 visible objects or attributes in the scene.",
]

results = []
for question in questions:
    question_messages = prefix_messages + [
        {"role": "user", "content": [{"type": "text", "text": question}]}
    ]

    full_inputs = processor.apply_chat_template(
        question_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    )
    full_inputs = {k: v.to(model.device) if hasattr(v, "to") else v for k, v in full_inputs.items()}

    delta_ids = full_inputs["input_ids"][:, prefix_len:]

    t1 = perf_counter()
    with torch.inference_mode():
        outputs = model.generate(
            input_ids=delta_ids,
            attention_mask=full_inputs["attention_mask"],
            past_key_values=deepcopy(base_cache),
            max_new_tokens=96,
            do_sample=False,
        )
    elapsed = perf_counter() - t1

    text = processor.batch_decode(
        outputs,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )[0]

    results.append({
        "question": question,
        "delta_tokens": int(delta_ids.shape[1]),
        "seconds": round(elapsed, 2),
        "answer": text,
    })

results

## 4. 結果を読みやすく表示

In [ ]:
for item in results:
    print('=' * 80)
    print('Question:', item['question'])
    print('Delta tokens:', item['delta_tokens'])
    print('Seconds:', item['seconds'])
    print(item['answer'])
    print()

## 5. 実務上のポイント

- 同じ画像・同じ共通文脈に対して**複数質問**する時に有効
- 最初の `prefill` は重いが、後続は **差分トークンだけ**で済む
- `deepcopy(base_cache)` を使うと、質問ごとに独立した分岐がしやすい
- 長い文脈では KV cache 自体のメモリ消費も増えるため、必要に応じて
  - offloaded cache
  - static cache
  - quantized cache
  を検討する価値があります

つまり、**画像を含む共有prefixを再利用したいなら、Qwen3-VLでもKV cacheは普通に有効**です。